# H-A-Alpha Wishart evaluation

In [ ]:
%load_ext autoreload
%autoreload 2

import os
# avoid thread conflicts between numpy and dask
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pathlib import Path
import xarray as xr
from polsarpro.io import open_netcdf_beam
from polsarpro.dev.io import read_psp_bin
from polsarpro.decompositions import h_a_alpha
from polsarpro.classification import wishart_h_a_alpha
import shutil

# optional import for progress bar
from dask.diagnostics import ProgressBar

# change to your local C-PolSARpro install dir
c_psp_dir = "/home/c_psp/Soft/bin/"
os.environ["PATH"]+=os.pathsep+f"{c_psp_dir}/data_process_sngl/"
os.environ["PATH"]+=os.pathsep+f"{c_psp_dir}/data_convert/"

# change to your data paths
# original dataset
input_alos_data = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")

# input files from C
input_test_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1/")

# output files from C
output_test_dir = Path("/data/psp/res/wishart_c")
if not os.path.isdir(output_test_dir):
    os.mkdir(output_test_dir)


def write_config_file(directory: Path, nrow: int, ncol: int) -> Path:
    """
    Create a config.txt file in the given directory with specified dimensions.

    Args:
        directory: Target directory.
        nrow: Number of rows.
        ncol: Number of columns.

    Returns:
        Path to the created config file.
    """
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)

    config_path = directory / "config.txt"

    content = f"""Nrow
{nrow}
---------
Ncol
{ncol}
---------
PolarCase
bistatic
---------
PolarType
full 
"""

    config_path.write_text(content)

    return config_path


# compute large image
# row1, row2 = 0, 18432
# col1, col2 = 0, 1248

# smaller window
# row1, row2 = 8000, 8500
row1, row2 = 8000, 12000
col1, col2 = 200, 1100

## Run the C-version on some test data
### Compute H-A-alpha

#### Note: 
We compute the decomposition on the whole image and then apply the window to the classification.  
This is to avoid offset problems (classification uses the same offsets on input image a decmposition results.)

In [ ]:
# only activate flags for H, A & Alpha

input_dict = dict(
    id=input_test_dir,
    od=output_test_dir,
    iodf="S2T3",
    nwr=7,
    nwc=7,
    ofr=0,
    ofc=0,
    fnr=18432,
    fnc=1248,
    fl1=0,
    fl2=0,
    fl3=1,
    fl4=1,
    fl5=1,
    fl6=0,
    fl7=0,
    fl8=0,
    fl9=0,
    errf="/tmp/MemoryAllocError.txt",
    mask=f"{input_test_dir}/mask_valid_pixels.bin",
)

# making input string: -param1 value1 -param2 value2 etc
param_str = " ".join([f"-{k} {input_dict[k]}" for k in input_dict])
cmd = f"h_a_alpha_decomposition.exe {param_str}"
print(cmd)
# os.system(cmd)

# CPSP does not create another config file in output dir
# shutil.copy(input_test_dir / "config.txt", output_test_dir)

# write_config_file(directory=output_test_dir, nrow=row2-row1, ncol=col2-col1)

### Compute Wishart using H-A-Alpha results

In [ ]:
input_dict = dict(
    id=input_test_dir,
    od=output_test_dir,
    iodf="S2",
    nwr=7,
    nwc=7,
    ofr=row1, # offsets are applied to input data and HAA results
    ofc=col1,
    fnr=row2-row1,
    fnc=col2-col1,
    hf=f"{output_test_dir}/entropy.bin",
    af=f"{output_test_dir}/anisotropy.bin",
    alf=f"{output_test_dir}/alpha.bin",
    nit="10",
    pct="10",
    bmp=0,
    errf="/tmp/MemoryAllocError.txt",
    mask=f"{input_test_dir}/mask_valid_pixels.bin",
)

# making input string: -param1 value1 -param2 value2 etc
param_str = " ".join([f"-{k} {input_dict[k]}" for k in input_dict])
cmd = f"wishart_h_a_alpha_classifier.exe {param_str}"
print(cmd)
# WARNING: this takes long!
# os.system(cmd)

# need adapted conf file to read in python
write_config_file(directory=output_test_dir, nrow=row2-row1, ncol=col2-col1)

## Load ALOS data and C outputs

In [ ]:
# uncomment to test on S matrix made with SNAP
S = open_netcdf_beam(input_alos_data).persist()#.chunk("auto")

flags = ("entropy", "anisotropy", "alpha")

dct = {}
for it in flags:
    dat = read_psp_bin(output_test_dir / f"haa_full/{it}.bin")
    dct[it] = (("y", "x"), dat)

haa = xr.Dataset(dct, attrs={"poltype":"h_a_alpha"})#.chunk("auto")
# haa

## Apply the decomposition

In [ ]:
file_out = "/data/psp/res/test_wishart_h_a_alpha.nc"
# netcdf writer cannot overwrite
if os.path.isfile(file_out):
    os.remove(file_out)

# to stay consistent with C, we first compute HAA 
# on the whole image then crop before Wishart
with ProgressBar():
    # haa = h_a_alpha(S, boxcar_size=[7, 7], flags=flags)#.chunk("auto")

    # haa = haa.isel(y=slice(row1, row2), x=slice(col1, col2))
    res = wishart_h_a_alpha(
        S.isel(y=slice(row1, row2), x=slice(col1, col2)),
        boxcar_size=[7, 7],
        max_iter=10,
        # tol_pct=None,
        tol_pct=10,
        h_a_alpha_result=haa.isel(y=slice(row1, row2), x=slice(col1, col2)),
    ).to_netcdf(file_out)

# Numerical evaluation

In [ ]:
# open python output (comment if using compute)
out_py = xr.open_dataset("/data/psp/res/test_wishart_h_a_alpha.nc")

# open C-PolSARPro outputs
out_c = {}

out_names = {
    "wishart_h_a_alpha_class": "wishart_H_A_alpha_class_7x7",
    "wishart_h_alpha_class": "wishart_H_alpha_class_7x7",
}

for name in out_names:
    file_name = output_test_dir / f"{out_names[name]}.bin"
    out_c[name] = read_psp_bin(file_name)

In [ ]:
var_name = "wishart_h_alpha_class"
arr_c = out_c[var_name]
arr_py = out_py[var_name]

import matplotlib.pyplot as plt
plt.figure(figsize=(15, 10))
plt.subplot(131)
plt.imshow((arr_c - arr_py)[::4], interpolation="none", cmap="Dark2")
plt.title("diff")
plt.subplot(132)
plt.imshow(arr_c[::4], interpolation="none", cmap="Dark2")
plt.colorbar(fraction=0.046, pad=0.04)
plt.title("c")
plt.subplot(133)
plt.imshow(arr_py[::4], interpolation="none", cmap="Dark2")
plt.colorbar(fraction=0.046, pad=0.04)
plt.title("py")

In [ ]:
var_name = "wishart_h_a_alpha_class"
arr_c = out_c[var_name]
arr_py = out_py[var_name]

import matplotlib.pyplot as plt
plt.figure(figsize=(15, 10))
plt.subplot(131)
plt.imshow((arr_c - arr_py)[::4], interpolation="none", cmap="tab20")
plt.colorbar(fraction=0.046, pad=0.04)
plt.title("diff")
plt.subplot(132)
plt.imshow(arr_c[::4], interpolation="none", cmap="tab20")
plt.colorbar(fraction=0.046, pad=0.04)
plt.title("c")
plt.subplot(133)
plt.imshow(arr_py[::4], interpolation="none", cmap="tab20")
plt.colorbar(fraction=0.046, pad=0.04)
plt.title("py")